## Load spark

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("WeatherAssignment") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.1.1


## Load datasets

In [8]:
import os

base_path = "."

files = []

for year in range(2015, 2025):

    cincinnati = f"{base_path}/{year}/72429793812.csv"
    florida = f"{base_path}/{year}/99495199999.csv"

    if os.path.exists(cincinnati):
        files.append((year, "Cincinnati", "72429793812", cincinnati))

    if os.path.exists(florida):
        files.append((year, "Florida", "99495199999", florida))

print("Total datasets found:", len(files))

for f in files:
    print(f)

Total datasets found: 19
(2015, 'Cincinnati', '72429793812', './2015/72429793812.csv')
(2015, 'Florida', '99495199999', './2015/99495199999.csv')
(2016, 'Cincinnati', '72429793812', './2016/72429793812.csv')
(2017, 'Cincinnati', '72429793812', './2017/72429793812.csv')
(2017, 'Florida', '99495199999', './2017/99495199999.csv')
(2018, 'Cincinnati', '72429793812', './2018/72429793812.csv')
(2018, 'Florida', '99495199999', './2018/99495199999.csv')
(2019, 'Cincinnati', '72429793812', './2019/72429793812.csv')
(2019, 'Florida', '99495199999', './2019/99495199999.csv')
(2020, 'Cincinnati', '72429793812', './2020/72429793812.csv')
(2020, 'Florida', '99495199999', './2020/99495199999.csv')
(2021, 'Cincinnati', '72429793812', './2021/72429793812.csv')
(2021, 'Florida', '99495199999', './2021/99495199999.csv')
(2022, 'Cincinnati', '72429793812', './2022/72429793812.csv')
(2022, 'Florida', '99495199999', './2022/99495199999.csv')
(2023, 'Cincinnati', '72429793812', './2023/72429793812.csv')
(202

In [9]:
dataset_counts = []

for year, city, station, path in files:

    df = spark.read.option("header", True).csv(path)

    count = df.count()

    dataset_counts.append((year, city, station, count))

dataset_counts

[(2015, 'Cincinnati', '72429793812', 365),
 (2015, 'Florida', '99495199999', 355),
 (2016, 'Cincinnati', '72429793812', 366),
 (2017, 'Cincinnati', '72429793812', 365),
 (2017, 'Florida', '99495199999', 283),
 (2018, 'Cincinnati', '72429793812', 365),
 (2018, 'Florida', '99495199999', 363),
 (2019, 'Cincinnati', '72429793812', 365),
 (2019, 'Florida', '99495199999', 345),
 (2020, 'Cincinnati', '72429793812', 366),
 (2020, 'Florida', '99495199999', 365),
 (2021, 'Cincinnati', '72429793812', 365),
 (2021, 'Florida', '99495199999', 104),
 (2022, 'Cincinnati', '72429793812', 365),
 (2022, 'Florida', '99495199999', 259),
 (2023, 'Cincinnati', '72429793812', 365),
 (2023, 'Florida', '99495199999', 276),
 (2024, 'Cincinnati', '72429793812', 366),
 (2024, 'Florida', '99495199999', 133)]

In [10]:
counts_df = spark.createDataFrame(
    dataset_counts,
    ["YEAR", "CITY", "STATION", "ROW_COUNT"]
)

counts_df.orderBy("YEAR", "CITY").show(25, truncate=False)

+----+----------+-----------+---------+
|YEAR|CITY      |STATION    |ROW_COUNT|
+----+----------+-----------+---------+
|2015|Cincinnati|72429793812|365      |
|2015|Florida   |99495199999|355      |
|2016|Cincinnati|72429793812|366      |
|2017|Cincinnati|72429793812|365      |
|2017|Florida   |99495199999|283      |
|2018|Cincinnati|72429793812|365      |
|2018|Florida   |99495199999|363      |
|2019|Cincinnati|72429793812|365      |
|2019|Florida   |99495199999|345      |
|2020|Cincinnati|72429793812|366      |
|2020|Florida   |99495199999|365      |
|2021|Cincinnati|72429793812|365      |
|2021|Florida   |99495199999|104      |
|2022|Cincinnati|72429793812|365      |
|2022|Florida   |99495199999|259      |
|2023|Cincinnati|72429793812|365      |
|2023|Florida   |99495199999|276      |
|2024|Cincinnati|72429793812|366      |
|2024|Florida   |99495199999|133      |
+----+----------+-----------+---------+



## Data cleaning

### Load and combine all CSVs

In [18]:
dfs = []

for year, city, station, path in files:
    df = spark.read.option("header", True).csv(path)
    dfs.append(df)

weather_df = dfs[0]

for df in dfs[1:]:
    weather_df = weather_df.union(df)

weather_df.printSchema()
weather_df.show(5, truncate=False)

root
 |-- STATION: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- LATITUDE: string (nullable = true)
 |-- LONGITUDE: string (nullable = true)
 |-- ELEVATION: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- TEMP: string (nullable = true)
 |-- TEMP_ATTRIBUTES: string (nullable = true)
 |-- DEWP: string (nullable = true)
 |-- DEWP_ATTRIBUTES: string (nullable = true)
 |-- SLP: string (nullable = true)
 |-- SLP_ATTRIBUTES: string (nullable = true)
 |-- STP: string (nullable = true)
 |-- STP_ATTRIBUTES: string (nullable = true)
 |-- VISIB: string (nullable = true)
 |-- VISIB_ATTRIBUTES: string (nullable = true)
 |-- WDSP: string (nullable = true)
 |-- WDSP_ATTRIBUTES: string (nullable = true)
 |-- MXSPD: string (nullable = true)
 |-- GUST: string (nullable = true)
 |-- MAX: string (nullable = true)
 |-- MAX_ATTRIBUTES: string (nullable = true)
 |-- MIN: string (nullable = true)
 |-- MIN_ATTRIBUTES: string (nullable = true)
 |-- PRCP: string (nullable = t

In [19]:
from pyspark.sql.functions import col, to_date, year, month, trim, lpad

weather_typed = weather_df \
    .withColumn("DATE", to_date(col("DATE"), "yyyy-MM-dd")) \
    .withColumn("LATITUDE", col("LATITUDE").cast("double")) \
    .withColumn("LONGITUDE", col("LONGITUDE").cast("double")) \
    .withColumn("ELEVATION", col("ELEVATION").cast("double")) \
    .withColumn("TEMP", col("TEMP").cast("double")) \
    .withColumn("DEWP", col("DEWP").cast("double")) \
    .withColumn("SLP", col("SLP").cast("double")) \
    .withColumn("STP", col("STP").cast("double")) \
    .withColumn("VISIB", col("VISIB").cast("double")) \
    .withColumn("WDSP", col("WDSP").cast("double")) \
    .withColumn("MXSPD", col("MXSPD").cast("double")) \
    .withColumn("GUST", col("GUST").cast("double")) \
    .withColumn("MAX", col("MAX").cast("double")) \
    .withColumn("MIN", col("MIN").cast("double")) \
    .withColumn("PRCP", col("PRCP").cast("double")) \
    .withColumn("SNDP", col("SNDP").cast("double")) \
    .withColumn("YEAR", year(col("DATE"))) \
    .withColumn("MONTH", month(col("DATE"))) \
    .withColumn("FRSHTT", lpad(trim(col("FRSHTT")), 6, "0"))

In [20]:
weather_typed.printSchema()
weather_typed.select(
    "STATION", "NAME", "DATE", "YEAR", "MONTH",
    "TEMP", "MAX", "MIN", "WDSP", "GUST", "PRCP", "FRSHTT"
).show(10, truncate=False)

root
 |-- STATION: string (nullable = true)
 |-- DATE: date (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- ELEVATION: double (nullable = true)
 |-- NAME: string (nullable = true)
 |-- TEMP: double (nullable = true)
 |-- TEMP_ATTRIBUTES: string (nullable = true)
 |-- DEWP: double (nullable = true)
 |-- DEWP_ATTRIBUTES: string (nullable = true)
 |-- SLP: double (nullable = true)
 |-- SLP_ATTRIBUTES: string (nullable = true)
 |-- STP: double (nullable = true)
 |-- STP_ATTRIBUTES: string (nullable = true)
 |-- VISIB: double (nullable = true)
 |-- VISIB_ATTRIBUTES: string (nullable = true)
 |-- WDSP: double (nullable = true)
 |-- WDSP_ATTRIBUTES: string (nullable = true)
 |-- MXSPD: double (nullable = true)
 |-- GUST: double (nullable = true)
 |-- MAX: double (nullable = true)
 |-- MAX_ATTRIBUTES: string (nullable = true)
 |-- MIN: double (nullable = true)
 |-- MIN_ATTRIBUTES: string (nullable = true)
 |-- PRCP: double (nullable = tru

In [21]:
valid_date = col("DATE").isNotNull()
valid_temp = col("TEMP") != 9999.9
valid_max = col("MAX") != 9999.9
valid_min = col("MIN") != 9999.9
valid_wdsp = col("WDSP") != 999.9
valid_gust = col("GUST") != 999.9
valid_prcp = col("PRCP") != 99.99

In [22]:
weather_base = weather_typed.filter(valid_date)

weather_base.select(
    "STATION", "NAME", "DATE", "YEAR", "MONTH", "TEMP", "MAX", "MIN"
).show(10, truncate=False)

+-----------+------------------------------------------------+----------+----+-----+----+----+----+
|STATION    |NAME                                            |DATE      |YEAR|MONTH|TEMP|MAX |MIN |
+-----------+------------------------------------------------+----------+----+-----+----+----+----+
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2015-01-01|2015|1    |26.1|39.0|15.1|
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2015-01-02|2015|1    |33.3|39.0|18.0|
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2015-01-03|2015|1    |41.8|61.0|26.1|
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2015-01-04|2015|1    |50.0|61.0|35.1|
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2015-01-05|2015|1    |20.2|33.1|14.0|
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2015-01-06|2015|1    |22.2|28.4|18.0|
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2015-01-07|2015|1    |14.6|28.0|7.0 |


## Questions from assignment

### Question 3: hottest day (MAX) for each year

In [23]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

hottest_base = weather_base.filter(valid_max)

window_year = Window.partitionBy("YEAR").orderBy(col("MAX").desc(), col("DATE").asc())

hottest_each_year = hottest_base \
    .withColumn("rn", row_number().over(window_year)) \
    .filter(col("rn") == 1) \
    .select("YEAR", "STATION", "NAME", "DATE", "MAX") \
    .orderBy("YEAR")

hottest_each_year.show(10, truncate=False)

+----+-----------+------------------------------------------------+----------+-----+
|YEAR|STATION    |NAME                                            |DATE      |MAX  |
+----+-----------+------------------------------------------------+----------+-----+
|2015|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2015-06-12|91.9 |
|2016|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2016-07-24|93.9 |
|2017|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2017-07-22|91.9 |
|2018|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2018-07-04|96.1 |
|2019|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2019-09-30|95.0 |
|2020|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2020-07-05|93.9 |
|2021|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2021-08-12|95.0 |
|2022|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2022-06-14|96.1 |
|2023|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH U

### Question 4: coldest March day across all years

In [24]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

coldest_march_base = weather_base.filter((col("MONTH") == 3) & valid_min)

window_all = Window.orderBy(col("MIN").asc(), col("DATE").asc())

coldest_march = coldest_march_base \
    .withColumn("rn", row_number().over(window_all)) \
    .filter(col("rn") == 1) \
    .select("STATION", "NAME", "DATE", "MIN")

coldest_march.show(truncate=False)

26/03/06 12:45:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 12:45:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 12:45:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+------------------------------------------------+----------+---+
|STATION    |NAME                                            |DATE      |MIN|
+-----------+------------------------------------------------+----------+---+
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2015-03-06|3.2|
+-----------+------------------------------------------------+----------+---+



### Question 5: Year with the most precipitation for Cincinnati and Florida.

In [29]:
from pyspark.sql.functions import avg, round, row_number
from pyspark.sql.window import Window

# remove invalid precipitation values
prcp_clean = weather_base.filter((col("PRCP").isNotNull()) & (col("PRCP") < 90))

mean_prcp_by_station_year = prcp_clean.groupBy("STATION", "NAME", "YEAR") \
    .agg(round(avg("PRCP"), 3).alias("MEAN_PRCP"))

window_station = Window.partitionBy("STATION").orderBy(col("MEAN_PRCP").desc())

most_precip_year = mean_prcp_by_station_year \
    .withColumn("rn", row_number().over(window_station)) \
    .filter(col("rn") == 1) \
    .select("STATION", "NAME", "YEAR", "MEAN_PRCP") \
    .orderBy("STATION")

most_precip_year.show(truncate=False)

+-----------+------------------------------------------------+----+---------+
|STATION    |NAME                                            |YEAR|MEAN_PRCP|
+-----------+------------------------------------------------+----+---------+
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2018|0.158    |
|99495199999|SEBASTIAN INLET STATE PARK, FL US               |2015|0.0      |
+-----------+------------------------------------------------+----+---------+



### Question 6: Percentage of missing GUST values for Cincinnati and Florida in 2024

In [31]:
from pyspark.sql.functions import count, sum as spark_sum, when, round

gust_2024 = weather_base.filter(col("YEAR") == 2024)

gust_missing = gust_2024.groupBy("STATION", "NAME").agg(
    count("*").alias("TOTAL_DAYS"),
    spark_sum(when(col("GUST") == 999.9, 1).otherwise(0)).alias("MISSING_GUST")
)

gust_missing = gust_missing.withColumn(
    "MISSING_PERCENT",
    round(col("MISSING_GUST") / col("TOTAL_DAYS") * 100, 2)
)

gust_missing.select("STATION","NAME","MISSING_PERCENT").show(truncate=False)

+-----------+------------------------------------------------+---------------+
|STATION    |NAME                                            |MISSING_PERCENT|
+-----------+------------------------------------------------+---------------+
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|39.07          |
|99495199999|SEBASTIAN INLET STATE PARK, FL US               |100.0          |
+-----------+------------------------------------------------+---------------+



### Question 7: Find the mean, median, mode, and standard deviation of the temperature (column TEMP) for Cincinnati in each month for the year 2020.

In [34]:
from pyspark.sql.functions import avg, stddev, round, expr, col, count, row_number
from pyspark.sql.window import Window

# Cincinnati 2020 data
cincy_2020 = weather_base.filter(
    (col("STATION") == "72429793812") &
    (col("YEAR") == 2020) &
    (col("TEMP") != 9999.9)
)

# mean, median, std
stats_df = cincy_2020.groupBy("MONTH").agg(
    round(avg("TEMP"),2).alias("MEAN_TEMP"),
    round(expr("percentile_approx(TEMP,0.5)"),2).alias("MEDIAN_TEMP"),
    round(stddev("TEMP"),2).alias("STD_TEMP")
)

# mode calculation
mode_counts = cincy_2020.groupBy("MONTH","TEMP").agg(count("*").alias("freq"))

window_mode = Window.partitionBy("MONTH").orderBy(col("freq").desc())

mode_df = mode_counts \
    .withColumn("rn", row_number().over(window_mode)) \
    .filter(col("rn") == 1) \
    .select("MONTH", col("TEMP").alias("MODE_TEMP"))

# combine everything
final_stats = stats_df.join(mode_df, "MONTH").orderBy("MONTH")

final_stats.show(12, truncate=False)

+-----+---------+-----------+--------+---------+
|MONTH|MEAN_TEMP|MEDIAN_TEMP|STD_TEMP|MODE_TEMP|
+-----+---------+-----------+--------+---------+
|1    |37.95    |37.7       |8.35    |24.7     |
|2    |36.59    |36.0       |7.9     |25.9     |
|3    |49.07    |47.8       |8.78    |43.0     |
|4    |51.78    |51.0       |7.31    |55.7     |
|5    |60.89    |63.7       |9.31    |73.9     |
|6    |72.55    |73.7       |4.9     |74.7     |
|7    |77.6     |77.9       |2.34    |77.5     |
|8    |73.35    |73.7       |3.49    |73.2     |
|9    |66.1     |65.8       |7.12    |75.3     |
|10   |55.19    |54.0       |6.73    |63.8     |
|11   |48.0     |47.7       |6.83    |47.7     |
|12   |35.99    |35.2       |6.64    |32.1     |
+-----+---------+-----------+--------+---------+



### Question 8: Find the top 10 days with the lowest Wind Chill for Cincinnati in 2017

In [35]:
from pyspark.sql.functions import pow

cincy_2017 = weather_base.filter(
    (col("STATION") == "72429793812") &
    (col("YEAR") == 2017) &
    (col("TEMP") < 50) &
    (col("WDSP") > 3) &
    (col("TEMP") != 9999.9) &
    (col("WDSP") != 999.9)
)

wind_chill_df = cincy_2017.withColumn(
    "WIND_CHILL",
    35.74
    + 0.6215 * col("TEMP")
    - 35.75 * pow(col("WDSP"), 0.16)
    + 0.4275 * col("TEMP") * pow(col("WDSP"), 0.16)
)

lowest_wc = wind_chill_df.select(
    "DATE", "TEMP", "WDSP", "WIND_CHILL"
).orderBy(col("WIND_CHILL").asc())

lowest_wc.show(10, truncate=False)

+----------+----+----+-------------------+
|DATE      |TEMP|WDSP|WIND_CHILL         |
+----------+----+----+-------------------+
|2017-01-07|10.5|7.0 |-0.4140156367932173|
|2017-12-31|11.0|5.3 |2.0339767075993116 |
|2017-12-27|13.0|5.8 |3.820645509123832  |
|2017-12-28|13.6|5.8 |4.533355269061226  |
|2017-01-06|13.6|5.5 |4.868933041653884  |
|2017-01-08|15.9|5.2 |7.929748208036862  |
|2017-12-25|25.8|13.5|14.285113218297408 |
|2017-12-30|21.6|5.3 |14.539211253038193 |
|2017-01-05|22.2|5.8 |14.748861828163854 |
|2017-12-26|23.3|6.2 |15.688977805634499 |
+----------+----+----+-------------------+
only showing top 10 rows


### Question 9: Investigate how many days had extreme weather conditions for Florida using the FRSHTT column 

In [36]:
from pyspark.sql.functions import substring, sum as spark_sum

florida_weather = weather_base.filter(col("STATION") == "99495199999")

extreme_weather = florida_weather.select(
    spark_sum(substring("FRSHTT",1,1).cast("int")).alias("FOG_DAYS"),
    spark_sum(substring("FRSHTT",2,1).cast("int")).alias("RAIN_DAYS"),
    spark_sum(substring("FRSHTT",3,1).cast("int")).alias("SNOW_DAYS"),
    spark_sum(substring("FRSHTT",4,1).cast("int")).alias("HAIL_DAYS"),
    spark_sum(substring("FRSHTT",5,1).cast("int")).alias("THUNDER_DAYS"),
    spark_sum(substring("FRSHTT",6,1).cast("int")).alias("TORNADO_DAYS")
)

extreme_weather.show()

+--------+---------+---------+---------+------------+------------+
|FOG_DAYS|RAIN_DAYS|SNOW_DAYS|HAIL_DAYS|THUNDER_DAYS|TORNADO_DAYS|
+--------+---------+---------+---------+------------+------------+
|       0|        0|        0|        0|           0|           0|
+--------+---------+---------+---------+------------+------------+



In [40]:
from pyspark.sql.functions import when, lit, substring, col, sum as spark_sum

florida_weather = weather_base.filter(col("STATION") == "99495199999")

extreme_days = florida_weather.withColumn(
    "HAS_EXTREME",
    when(col("FRSHTT") != "000000", lit(1)).otherwise(lit(0))
)

extreme_days.select(spark_sum("HAS_EXTREME").alias("EXTREME_WEATHER_DAYS")).show()

+--------------------+
|EXTREME_WEATHER_DAYS|
+--------------------+
|                   0|
+--------------------+



### Question 10: Predict the maximum Temperature for Cincinnati for November and December 2024, based on the previous 2 years of weather data

In [39]:
from pyspark.sql.functions import col, max as spark_max
import builtins

# Cincinnati monthly maximum temperatures
cincy_max = weather_base.filter(
    (col("STATION") == "72429793812") & (col("MAX") != 9999.9)
)

monthly_max = cincy_max.groupBy("YEAR", "MONTH").agg(
    spark_max("MAX").alias("MONTHLY_MAX")
)

# keep only Nov and Dec for 2022 and 2023
nov_dec = monthly_max.filter(
    (col("MONTH").isin([11, 12])) & (col("YEAR").isin([2022, 2023]))
).orderBy("MONTH", "YEAR")

# collect values
rows = nov_dec.collect()
data = {(r["YEAR"], r["MONTH"]): r["MONTHLY_MAX"] for r in rows}

# simple linear-trend prediction for 2024
predictions = []
for month in [11, 12]:
    y2022 = data[(2022, month)]
    y2023 = data[(2023, month)]
    pred_2024 = y2023 + (y2023 - y2022)
    predictions.append((2024, month, y2022, y2023, builtins.round(pred_2024, 2)))

# show results
pred_df = spark.createDataFrame(
    predictions,
    ["YEAR", "MONTH", "MAX_2022", "MAX_2023", "PREDICTED_MAX_2024"]
)

pred_df.show(truncate=False)

+----+-----+--------+--------+------------------+
|YEAR|MONTH|MAX_2022|MAX_2023|PREDICTED_MAX_2024|
+----+-----+--------+--------+------------------+
|2024|11   |75.9    |80.1    |84.3              |
|2024|12   |66.0    |64.0    |62.0              |
+----+-----+--------+--------+------------------+



In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

# Ensure the data_plots directory exists
os.makedirs('data_plots', exist_ok=True)

def plot_and_save(csv_path, plot_name):
    df = pd.read_csv(csv_path)
    plt.figure(figsize=(10, 4))
    # Example: plot the first two columns if they exist
    if df.shape[1] >= 2:
        plt.plot(df.iloc[:, 0], df.iloc[:, 1], label=f'{os.path.basename(csv_path)}')
        plt.xlabel(df.columns[0])
        plt.ylabel(df.columns[1])
        plt.title(f'{os.path.basename(csv_path)}: {df.columns[1]} vs {df.columns[0]}')
        plt.legend()
        plt.tight_layout()
        plt.savefig(f'data_plots/{plot_name}.png')
        plt.close()
    else:
        print(f'Skipped {csv_path}: not enough columns to plot')

# Example usage for one file:
# plot_and_save('2015/72429793812.csv', '2015_72429793812')

# Batch process all CSVs in all year folders
years = [str(y) for y in range(2015, 2025)]
for year in years:
    for fname in os.listdir(year):
        if fname.endswith('.csv'):
            csv_path = os.path.join(year, fname)
            plot_name = f'{year}_{fname.replace(".csv", "")}'
            plot_and_save(csv_path, plot_name)
print('Plots saved in data_plots/')